In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [4]:
import json


def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
    each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task",
        },
        ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [ ]:
# print the dataset
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URL. The function should take a URL like 's3://my-bucket.s3.us-west-2.amazonaws.com/key' and return 'us-west-2'."},
 {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'."},
 {'task': "Write a regular expression that validates AWS EC2 security group IDs, which follow the pattern 'sg-' followed by exactly 17 hexadecimal characters."}]

In [7]:
# save the dataset to a JSON file
dataset = generate_dataset()
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [9]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results


with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))


[
  {
    "output": "# AWS Region Extraction from S3 Bucket URL\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\nfrom urllib.parse import urlparse\n\ndef extract_region_from_s3_url(s3_url: str) -> str:\n    \"\"\"\n    Extract the AWS region from an S3 bucket URL.\n    \n    Supports formats:\n    - s3://bucket-name.region.amazonaws.com\n    - https://bucket-name.s3.region.amazonaws.com\n    - https://s3.region.amazonaws.com/bucket-name\n    \n    Args:\n        s3_url (str): The S3 bucket URL\n        \n    Returns:\n        str: The AWS region (e.g., 'us-east-1')\n        \n    Raises:\n        ValueError: If the region cannot be extracted from the URL\n    \"\"\"\n    if not s3_url:\n        raise ValueError(\"URL cannot be empty\")\n    \n    # Pattern for: bucket-name.region.amazonaws.com\n    pattern1 = r'\\.([a-z0-9\\-]+)\\.amazonaws\\.com'\n    \n    # Pattern for: s3.region.amazonaws.com\n    pattern2 = r's3\\.([a-z0-9\\-